# SpeechSense — управляющий запуск Kaggle
Сначала проверка данных и smoke. Полное обучение включается явно. GPU и Internet нужны для загрузки моделей. Test не используется при подборе.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT=Path("/kaggle/working/SpeechSense")
if not ROOT.exists(): subprocess.run(["git","clone","https://github.com/SamDerden/SpeechSense.git",str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,"-m","pip","install","-r","requirements-kaggle.txt"],check=True)


После установки при необходимости перезапустите kernel. PyTorch используется из GPU-образа Kaggle; версия записывается в отчёт.

In [ ]:
import os,sys,subprocess
from pathlib import Path
ROOT=Path("/kaggle/working/SpeechSense")
os.chdir(ROOT)
files=list(Path("/kaggle/input").rglob("merged_sentiment_dataset.csv.gz"))
assert len(files)==1, f"Expected one dataset, found {files}"
DATA=files[0]
OUT=Path("/kaggle/working/speechsense_runs")
OUT.mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT/"src"))
from dataset import splits,verify_splits
import json,torch
assert torch.cuda.is_available(), "Enable GPU"
manifest=verify_splits(splits(DATA))
manifest["torch"]=torch.__version__
manifest["commit"]=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print(manifest)


In [ ]:
def run(model, technique, smoke=True, seed=42, extra=()):
    dest=OUT/("smoke" if smoke else "full")/f"{model}_{technique}_{seed}"
    subprocess.run([sys.executable,"src/transformer_runner.py","--data",str(DATA),"--out",str(dest),"--model",model,"--technique",technique,"--seed",str(seed),*(["--smoke"] if smoke else []),*extra],check=True)
    return dest
run("tiny","baseline")


In [ ]:
RUN_FULL=False
TECHNIQUES=["baseline","weighted","differential","smoothing","clipping","custom_head","mean_pool","mixup","fgm","curriculum","multitask","active","back_translation","contextual"]
if RUN_FULL:
    for model in ["tiny","rubert","sentiment","xlmr","large"]: run(model,"baseline",False)
    for technique in TECHNIQUES[1:]: run("rubert",technique,False)


Distillation требует teacher logits с row_id и l0/l1/l2; ансамбли и пороги запускаются отдельным src/transformer_ensemble.py после получения предсказаний. Эти этапы не автоматизированы. Классические и нейросетевые ноутбуки пока запускаются отдельно. Финальный test только после фиксации выбора модели.